# 02. 실습: Controllable Thinking Effort 비용 곡선

목표: effort setting이 성능, 생성 토큰 수, 비용, latency 사이의 tradeoff를 어떻게 만드는지 시뮬레이션합니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Effort curve 정의

아래 숫자는 공식 benchmark 재현이 아니라 학습용 toy curve입니다. 낮은 effort는 빠르고 싸지만 성능이 낮고, 높은 effort는 더 많은 토큰을 써 성능을 올립니다.

In [ ]:
effort_curve = [
    {"effort": 0.20, "score": 0.42, "mean_tokens": 1800},
    {"effort": 0.40, "score": 0.52, "mean_tokens": 3600},
    {"effort": 0.60, "score": 0.60, "mean_tokens": 6800},
    {"effort": 0.80, "score": 0.66, "mean_tokens": 12000},
    {"effort": 0.99, "score": 0.70, "mean_tokens": 21000},
]

price_per_million_tokens = 2.40
tokens_per_second = 450


def add_cost_latency(row):
    cost = row["mean_tokens"] / 1_000_000 * price_per_million_tokens
    latency = row["mean_tokens"] / tokens_per_second
    return {**row, "cost_usd": cost, "latency_s": latency}


rows = [add_cost_latency(row) for row in effort_curve]
print("effort | score | tokens | cost_usd | latency_s")
print("--- | --- | --- | --- | ---")
for row in rows:
    print(f"{row['effort']:.2f} | {row['score']:.2f} | {row['mean_tokens']} | {row['cost_usd']:.4f} | {row['latency_s']:.1f}")

## 2. 업무별 effort 선택기

실무에서는 항상 최고 effort가 답이 아닙니다. latency, 비용, 최소 점수를 동시에 만족하는 가장 낮은 effort를 고르는 정책이 유용합니다.

In [ ]:
def choose_effort(min_score, max_latency_s, max_cost_usd):
    feasible = [
        row for row in rows
        if row["score"] >= min_score and row["latency_s"] <= max_latency_s and row["cost_usd"] <= max_cost_usd
    ]
    if not feasible:
        return None
    return min(feasible, key=lambda row: row["mean_tokens"])


workloads = [
    {"name": "chat autocomplete", "min_score": 0.45, "max_latency_s": 10, "max_cost_usd": 0.02},
    {"name": "coding agent step", "min_score": 0.60, "max_latency_s": 30, "max_cost_usd": 0.04},
    {"name": "hard benchmark run", "min_score": 0.68, "max_latency_s": 90, "max_cost_usd": 0.08},
]

for workload in workloads:
    choice = choose_effort(workload["min_score"], workload["max_latency_s"], workload["max_cost_usd"])
    if choice:
        print(f"{workload['name']}: effort={choice['effort']:.2f}, score={choice['score']:.2f}, tokens={choice['mean_tokens']}")
    else:
        print(f"{workload['name']}: no feasible effort under constraints")

## 3. Token efficiency 비교

모델 발표에서 중요한 것은 한 점의 benchmark score만이 아니라 score 대비 token 비용입니다. 아래는 목표 점수에 도달하는 데 필요한 token 수를 비교하는 간단한 방식입니다.

In [ ]:
competitors = [
    {"model": "Inkling effort sweep", "score": 0.60, "tokens": 6800},
    {"model": "Model A default", "score": 0.60, "tokens": 20000},
    {"model": "Model B high", "score": 0.65, "tokens": 32000},
]

baseline_tokens = competitors[1]["tokens"]
for item in competitors:
    relative = item["tokens"] / baseline_tokens
    print(f"{item['model']:20s} score={item['score']:.2f} tokens={item['tokens']:5d} relative_tokens={relative:.2f}x")